# Jalon 3 · Moteur d’optimisation des livraisons

Ce notebook lance le moteur sur les données du projet, puis montre les profils, les camions et les économies calculés. Le mode pilote s’exécute sur un couple avec et sans imputation. Le mode complet relance tous les couples et écrit dans un dossier distinct du plan figé.

[Guide du projet](../README.md) · [Bilan et validation du jalon 4](../milestone4/bilan_avant_apres.ipynb)

## 1. Choisir le périmètre

Chaque couple fournisseur × usine est optimisé séparément. Chaque ligne est entière et son poids et son volume décrivent la charge totale. Le moteur charge le jeu préparé du commit `a0e8aba`. Il résout le périmètre avec imputation et le périmètre sans imputation chiffrable.

In [1]:
import os
from pathlib import Path
import subprocess
import sys

import pandas as pd

mode = "pilote"  # "pilote" ou "complet"
couple = "SUP_00112:PLT_004"

root = Path.cwd()
if root.name == "milestone3":
    root = root.parent
engine = root / "milestone4" / "optimisation_cout.py"
published = root / "milestone4" / "resultats_verifies"
assert engine.is_file(), "Ouvrir le notebook depuis le dépôt ou le dossier milestone3."
assert mode in {"pilote", "complet"}

output = root / "results" / f"moteur-notebook-{mode}"
workers = 2 if mode == "pilote" else min(7, os.cpu_count() or 1)
env = {
    **os.environ,
    "PYTHONPATH": str(root / "src"),
    "OMP_NUM_THREADS": "1",
    "OPENBLAS_NUM_THREADS": "1",
}
print(f"{mode.capitalize()} · {couple if mode == 'pilote' else 'tous les couples'} · {workers} processus")

Pilote · SUP_00112:PLT_004 · 2 processus


## 2. Résoudre les dates et les chargements

Le solveur choisit le profil hebdomadaire, les jours de départ et les chargements selon la fenêtre J−7 à J, les jours ouvrés et les deux capacités. Chaque camion est valorisé séparément avec la grille poids-distance. Le modèle cherche le coût le plus bas. Deux camions peuvent être préférables si la grille les rend moins chers. Quand toutes les lignes éligibles un jour tiennent dans un camion, le moteur élimine les répartitions dont le tarif ne peut pas battre leur regroupement.

Le mode complet utilise le plan vérifié comme point de départ. Il consolide ensuite le meilleur coût faisable et la meilleure borne par couple avec ceux du plan publié. Les fenêtres de temps et la relaxation de capacité apportent des bornes utiles sur les couples difficiles. Un résultat à budget limité peut fournir un meilleur plan ou une meilleure borne.

In [2]:
def execute(command):
    with subprocess.Popen(command, cwd=root, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end="")
        if process.wait():
            raise RuntimeError("Le calcul a échoué. La sortie ci-dessus indique la cause.")

calculation = output if mode == "pilote" else output / "calcul"
command = [
    sys.executable, str(engine), "--earliest", "2026-11-02", "--adaptive",
    "--tight-window-cuts", "--workers", str(workers), "--output", str(calculation),
]
if mode == "pilote":
    command += ["--couple", couple, "--exact-size", "10", "--bound-seconds", "20"]
else:
    command += ["--exact-size", "0", "--bound-seconds", "12",
                "--incumbent", str(published / "affectations.csv")]

execute(command)
if mode == "complet":
    floors = output / "bornes_fenetres.csv"
    columns = ["scope", "supplier_id", "plant", "forced", "sans_calendrier", "fenetres_separees"]
    pd.read_csv(published / "couples.csv", usecols=columns).to_csv(floors, index=False)
    execute([
        sys.executable, str(engine.with_name("retenir_meilleurs_couts.py")),
        "--earliest", "2026-11-02", "--run", str(published), "--run", str(calculation),
        "--floors", str(floors), "--output", str(output),
    ])

2 independent models for 2 pair/scope combinations
{
  "avec_imputation": {
    "lines": 10,
    "groups": 1,
    "certified": 1,
    "exact_models": 1,
    "trucks": 6,
    "earliest_allowed": "2026-11-02",
    "early_trucks": 0,
    "early_lines": 0,
    "reference_eur": 1872.41,
    "lower_eur": 1389.27,
    "cost_eur": 1389.27,
    "saving_eur": 483.13,
    "gap_eur": 0.0
  },
  "sans_imputation_tarifable": {
    "lines": 9,
    "groups": 1,
    "certified": 1,
    "exact_models": 1,
    "trucks": 5,
    "earliest_allowed": "2026-11-02",
    "early_trucks": 1,
    "early_lines": 2,
    "reference_eur": 1611.98,
    "lower_eur": 1128.85,
    "cost_eur": 1128.85,
    "saving_eur": 483.13,
    "gap_eur": 0.0
  }
}


## 3. Lire le résultat

Le coût faisable vient des camions exportés. La borne inférieure donne le coût sous lequel le modèle ne peut pas descendre. Leur écart mesure la marge d’optimisation encore ouverte. Les résultats ci-dessous sont ceux de l’exécution du notebook. Le bilan de référence publié reste consultable dans les exports figés.

In [3]:
import json

report = "bilan.json" if mode == "pilote" else "comparatif.json"
summary = pd.DataFrame.from_dict(json.loads((output / report).read_text(encoding="utf-8")), orient="index")
metrics = ["lines", "trucks", "reference_eur", "cost_eur", "saving_eur", "gap_eur", "certified"]
print(summary[metrics].to_string())

couples = pd.read_csv(output / "couples.csv", dtype={"profile": str})
print()
print(couples[["scope", "supplier_id", "plant", "profile", "trucks", "certified"]].head(10).to_string(index=False))
print()
print("Aperçu des camions calculés")
trucks = pd.read_csv(output / "camions.csv")
print(trucks[["scope", "date", "truck", "lines", "weight", "volume", "cost_eur"]].head(6).round(2).to_string(index=False))
print()
for name in ("couples.csv", "camions.csv", "affectations.csv", report):
    path = output / name
    assert path.is_file(), name
    print(path.relative_to(root))

                           lines  trucks  reference_eur  cost_eur  saving_eur  gap_eur  certified
avec_imputation               10       6        1872.41   1389.27      483.13      0.0          1
sans_imputation_tarifable      9       5        1611.98   1128.85      483.13      0.0          1

                    scope supplier_id   plant profile  trucks  certified
          avec_imputation   SUP_00112 PLT_004     0 1       6       True
sans_imputation_tarifable   SUP_00112 PLT_004     1 3       5       True

Aperçu des camions calculés
          scope       date  truck  lines  weight  volume  cost_eur
avec_imputation 2026-11-09      1      2  6039.9    2.28    260.43
avec_imputation 2026-11-10      2      1 24500.0  110.40    260.43
avec_imputation 2026-11-16      3      2  3187.0    2.28    233.16
avec_imputation 2026-11-24      4      2  3187.0    2.28    233.16
avec_imputation 2026-12-08      5      2  3187.0    2.28    233.16
avec_imputation 2026-12-14      6      1  1593.5    1.1